# Observation-level baseline comparison: Standard QML vs PAQFE-v2

Matched 80/20 observation-level split; five initialization seeds (42–46). This is not unseen-composition validation. Both models use the same train/test rows, with scalers fitted on training rows only.


In [ ]:
!pip install pennylane

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 31.3 MB/s eta 0:00:00
  Attempting uninstall: autograd
    Found existing installation: autograd 1.9.1
    Uninstalling autograd-1.9.1:
      Successfully uninstalled autograd-1.9.1


In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import pennylane as qml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEEDS = [42, 43, 44, 45, 46]
TEST_SIZE = 0.20
SPLIT_RANDOM_STATE = 42
N_QUBITS = 10
N_LAYERS = 3
LR = 0.02
N_EPOCHS = 300
STANDARD_FEATURE_ORDER = ['xAl','VEC','dHmix','dSmix','delta','Omega','dchi','Mean_Tm','Temperature_C','log10_Strain_rate']
TARGET = 'UTS_MPa'
composition_cols = ['Al_norm','Co_norm','Cr_norm','Fe_norm','Ni_norm']


## Load the same Excel workbook and `B_Model_Ready` sheet.

In [ ]:
from google.colab import files
uploaded = files.upload()
file_path = list(uploaded.keys())[0]
df = pd.read_excel(file_path, sheet_name='B_Model_Ready')
cols = STANDARD_FEATURE_ORDER + [TARGET] + composition_cols
df[cols] = df[cols].apply(pd.to_numeric, errors='coerce')
df = df.dropna(subset=cols).copy()
df['Composition_Group'] = df[composition_cols].round(3).astype(str).agg('_'.join, axis=1)
print('Dataset shape:', df.shape)
print('Observations:', len(df))
print('Unique compositions:', df['Composition_Group'].nunique())


Saving PAQFE_UTS_Dataset_Corrected (1).xlsx to PAQFE_UTS_Dataset_Corrected (1).xlsx
Dataset shape: (49, 26)
Observations: 49
Unique compositions: 14


In [ ]:
train_df, test_df = train_test_split(df, test_size=TEST_SIZE, random_state=SPLIT_RANDOM_STATE)
X_train_raw = train_df[STANDARD_FEATURE_ORDER].to_numpy(dtype=np.float64)
X_test_raw = test_df[STANDARD_FEATURE_ORDER].to_numpy(dtype=np.float64)
y_train_raw = train_df[TARGET].to_numpy(dtype=np.float64)
y_test_raw = test_df[TARGET].to_numpy(dtype=np.float64)
x_scaler = StandardScaler()
X_train = x_scaler.fit_transform(X_train_raw)
X_test = x_scaler.transform(X_test_raw)
y_scaler = StandardScaler()
y_train = y_scaler.fit_transform(y_train_raw.reshape(-1,1)).ravel()
print('Train:', len(X_train), 'Test:', len(X_test))


Train: 39 Test: 10


In [ ]:
def build_standard_qml():
    dev = qml.device("default.qubit", wires=N_QUBITS)

    @qml.qnode(dev, interface="torch", diff_method="backprop")
    def circuit(inputs, weights):
        qml.templates.AngleEmbedding(
            inputs,
            wires=range(N_QUBITS),
            rotation="Y"
        )

        qml.templates.StronglyEntanglingLayers(
            weights,
            wires=range(N_QUBITS)
        )

        return [
            qml.expval(qml.PauliZ(i))
            for i in range(N_QUBITS)
        ]

    weight_shape = qml.StronglyEntanglingLayers.shape(
        n_layers=N_LAYERS,
        n_wires=N_QUBITS
    )

    class StandardQML(nn.Module):
        def __init__(self):
            super().__init__()

            self.weights = nn.Parameter(
                0.01 * torch.randn(
                    weight_shape,
                    dtype=torch.float64
                )
            )

            self.head = nn.Linear(
                N_QUBITS,
                1,
                dtype=torch.float64
            )

        def forward(self, x):
          q = circuit(x, self.weights)
          q = torch.stack(q, dim=1)
          return self.head(q).squeeze(-1)

    return StandardQML()

In [ ]:
Xtr = torch.tensor(X_train, dtype=torch.float64)
ytr = torch.tensor(y_train, dtype=torch.float64)
Xte = torch.tensor(X_test, dtype=torch.float64)
seed_results = []
standard_models_by_seed = {}

for seed in SEEDS:
    np.random.seed(seed)
    torch.manual_seed(seed)
    model = build_standard_qml()
    # Keep each trained seed model for held-out explainability.
    standard_models_by_seed[seed] = model
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    loss_fn = nn.MSELoss()
    model.train()
    for epoch in range(N_EPOCHS):
        optimizer.zero_grad()
        pred = model(Xtr)
        loss = loss_fn(pred, ytr)
        loss.backward()
        optimizer.step()
    model.eval()
    with torch.no_grad():
        pred_std = model(Xte).cpu().numpy()
    pred = y_scaler.inverse_transform(pred_std.reshape(-1,1)).ravel()
    mae = mean_absolute_error(y_test_raw, pred)
    rmse = np.sqrt(mean_squared_error(y_test_raw, pred))
    r2 = r2_score(y_test_raw, pred)
    seed_results.append({'Seed':seed, 'MAE':mae, 'RMSE':rmse, 'R2':r2})
    print(f'Seed {seed}: MAE={mae:.3f}, RMSE={rmse:.3f}, R2={r2:.3f}')


In [ ]:
seed_results_df = pd.DataFrame(seed_results)
summary = pd.DataFrame({
    'Metric':['MAE','RMSE','R2'],
    'Mean':[seed_results_df.MAE.mean(), seed_results_df.RMSE.mean(), seed_results_df.R2.mean()],
    'SD':[seed_results_df.MAE.std(ddof=1), seed_results_df.RMSE.std(ddof=1), seed_results_df.R2.std(ddof=1)]
})
display(seed_results_df)
display(summary)
seed_results_df.to_csv('Standard_QML_seed_stability_results.csv', index=False)
summary.to_csv('Standard_QML_seed_stability_summary.csv', index=False)


,Seed,MAE,RMSE,R2
0,42,216.836257,270.663677,0.242573
1,43,218.040359,273.360346,0.227405
2,44,208.490935,260.830249,0.296609
3,45,220.268308,276.665020,0.208612
4,46,223.512661,280.518810,0.186411


,Metric,Mean,SD
0,MAE,217.429704,5.605504
1,RMSE,272.407620,7.448178
2,R2,0.232322,0.041629


## PAQFE-v2 model and five-seed training

In [ ]:
GROUPED_FEATURE_ORDER = ["xAl","VEC","dchi","dHmix","dSmix","Omega","Mean_Tm","delta","Temperature_C","log10_Strain_rate"]
STRONG_RELATIONSHIP_DESCRIPTORS = {"delta", "Omega", "xAl", "VEC"}
UPWEIGHT_FACTOR = 1.2
LAMBDA_PRIOR = 0.01
QUBIT_BLOCKS = {"composition":[0,1,2], "thermodynamic":[3,4,5,6], "structural":[7], "context":[8,9]}

def ring_entangle(wires):
    if len(wires) < 2: return
    for i in range(len(wires)):
        qml.CNOT(wires=[wires[i], wires[(i+1) % len(wires)]])

def physics_entangling_layer():
    ring_entangle(QUBIT_BLOCKS["composition"])
    ring_entangle(QUBIT_BLOCKS["thermodynamic"])
    ring_entangle(QUBIT_BLOCKS["context"])
    fmap = {f:i for i,f in enumerate(GROUPED_FEATURE_ORDER)}
    qml.CNOT(wires=[fmap["delta"], fmap["Omega"]])
    qml.CNOT(wires=[fmap["xAl"], fmap["VEC"]])

physics_prior = torch.tensor([UPWEIGHT_FACTOR if f in STRONG_RELATIONSHIP_DESCRIPTORS else 1.0 for f in GROUPED_FEATURE_ORDER], dtype=torch.float64)

def build_paqfe_v2():
    dev = qml.device("default.qubit", wires=N_QUBITS)
    @qml.qnode(dev, interface="torch", diff_method="backprop")
    def circuit(inputs, weights):
        qml.templates.AngleEmbedding(inputs, wires=range(N_QUBITS), rotation="Y")
        physics_entangling_layer()
        qml.templates.StronglyEntanglingLayers(weights, wires=range(N_QUBITS))
        return [qml.expval(qml.PauliZ(i)) for i in range(N_QUBITS)]
    weight_shape = qml.StronglyEntanglingLayers.shape(n_layers=N_LAYERS, n_wires=N_QUBITS)
    class PAQFEv2(nn.Module):
        def __init__(self):
            super().__init__()
            self.alpha = nn.Parameter(physics_prior.clone())
            self.weights = nn.Parameter(0.01 * torch.randn(weight_shape, dtype=torch.float64))
            self.head = nn.Linear(N_QUBITS, 1, dtype=torch.float64)
        def forward(self, x):
            q = torch.stack(circuit(x * self.alpha, self.weights), dim=1)
            return self.head(q).squeeze(-1)
    return PAQFEv2()

paqfe_x_scaler = StandardScaler()
X_train_paqfe = paqfe_x_scaler.fit_transform(train_df[GROUPED_FEATURE_ORDER].to_numpy(dtype=np.float64))
X_test_paqfe = paqfe_x_scaler.transform(test_df[GROUPED_FEATURE_ORDER].to_numpy(dtype=np.float64))
paqfe_y_scaler = StandardScaler()
y_train_paqfe = paqfe_y_scaler.fit_transform(y_train_raw.reshape(-1,1)).ravel()
Xtr_paqfe = torch.tensor(X_train_paqfe, dtype=torch.float64)
Xte_paqfe = torch.tensor(X_test_paqfe, dtype=torch.float64)
ytr_paqfe = torch.tensor(y_train_paqfe, dtype=torch.float64)
paqfe_models_by_seed = {}
paqfe_seed_results = []
for seed in SEEDS:
    np.random.seed(seed); torch.manual_seed(seed)
    model = build_paqfe_v2()
    optimizer = torch.optim.Adam([{"params":[model.alpha],"lr":0.005}, {"params":[model.weights],"lr":LR}, {"params":model.head.parameters(),"lr":LR}])
    loss_fn = nn.MSELoss()
    model.train()
    for epoch in range(N_EPOCHS):
        optimizer.zero_grad()
        loss = loss_fn(model(Xtr_paqfe), ytr_paqfe) + LAMBDA_PRIOR * torch.mean((model.alpha-physics_prior)**2)
        loss.backward(); optimizer.step()
    model.eval(); paqfe_models_by_seed[seed] = model
    with torch.no_grad(): pred_std = model(Xte_paqfe).cpu().numpy()
    pred = paqfe_y_scaler.inverse_transform(pred_std.reshape(-1,1)).ravel()
    row = {"Seed":seed,"MAE":mean_absolute_error(y_test_raw,pred),"RMSE":np.sqrt(mean_squared_error(y_test_raw,pred)),"R2":r2_score(y_test_raw,pred)}
    paqfe_seed_results.append(row)
    print(f"PAQFE-v2 seed {seed}: MAE={row['MAE']:.3f}, RMSE={row['RMSE']:.3f}, R2={row['R2']:.3f}")
paqfe_seed_results_df = pd.DataFrame(paqfe_seed_results)
display(paqfe_seed_results_df)
display(paqfe_seed_results_df[['MAE','RMSE','R2']].agg(['mean','std']))
paqfe_seed_results_df.to_csv('PAQFE_v2_observation_level_seed_results.csv', index=False)


## Explainability — permutation importance on the fixed held-out test split

This retrains nothing: it uses the stored seed-42 models. Importance is the increase in held-out MAE (MPa) after permuting one feature. With a small test set, interpret rankings as exploratory.

In [ ]:
from sklearn.metrics import mean_absolute_error
import matplotlib.pyplot as plt

def permutation_importance_for_model(model, X_test_scaled, y_true, feature_names, inverse_y_scaler, repeats=30, random_state=42):
    rng = np.random.default_rng(random_state)
    X = np.asarray(X_test_scaled, dtype=np.float64)
    model.eval()
    def predict(Xarr):
        with torch.no_grad():
            z = model(torch.tensor(Xarr, dtype=torch.float64)).cpu().numpy().reshape(-1,1)
        return inverse_y_scaler.inverse_transform(z).ravel()
    baseline = mean_absolute_error(y_true, predict(X))
    rows=[]
    for j, name in enumerate(feature_names):
        deltas=[]
        for _ in range(repeats):
            Xp=X.copy(); Xp[:,j]=rng.permutation(Xp[:,j])
            deltas.append(mean_absolute_error(y_true,predict(Xp))-baseline)
        rows.append({"Feature":name,"MAE_increase_mean":float(np.mean(deltas)),"MAE_increase_sd":float(np.std(deltas,ddof=1))})
    return pd.DataFrame(rows).sort_values('MAE_increase_mean',ascending=False)

std_imp = permutation_importance_for_model(standard_models_by_seed[42], X_test, y_test_raw, STANDARD_FEATURE_ORDER, y_scaler)
paqfe_imp = permutation_importance_for_model(paqfe_models_by_seed[42], X_test_paqfe, y_test_raw, GROUPED_FEATURE_ORDER, paqfe_y_scaler)
print('Standard QML — seed 42'); display(std_imp)
print('PAQFE-v2 — seed 42'); display(paqfe_imp)
std_imp.to_csv('Standard_QML_seed42_permutation_importance.csv',index=False)
paqfe_imp.to_csv('PAQFE_v2_seed42_permutation_importance.csv',index=False)
fig, axes = plt.subplots(1,2,figsize=(14,5))
for ax, table, title in [(axes[0],std_imp,'Standard QML'),(axes[1],paqfe_imp,'PAQFE-v2')]:
    t=table.sort_values('MAE_increase_mean')
    ax.barh(t['Feature'],t['MAE_increase_mean'],xerr=t['MAE_increase_sd'])
    ax.set_title(title); ax.set_xlabel('Increase in held-out MAE (MPa)')
plt.tight_layout(); plt.show()
alpha_table = pd.DataFrame({'Feature':GROUPED_FEATURE_ORDER,'Physics_prior':physics_prior.numpy(),'Learned_alpha_seed42':paqfe_models_by_seed[42].alpha.detach().cpu().numpy()})
alpha_table['Change_from_prior'] = alpha_table['Learned_alpha_seed42'] - alpha_table['Physics_prior']
display(alpha_table)
alpha_table.to_csv('PAQFE_v2_seed42_alpha_explainability.csv',index=False)
